In [49]:
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from scipy.optimize import minimize_scalar
from scipy.stats import betabinom, chi2

In [3]:
import pandas as pd

In [4]:
url = URL.create(
    "postgresql+psycopg2",
    username="postgres",                      
    password=getpass("Postgres password: "),  
    host="localhost",
    port=5432,
    database="lol_worlds",
)
engine = create_engine(url)

# quick connection test
pd.read_sql("SELECT 1 AS ok", engine)

Postgres password:  ········


,ok
0,1


# Part 1. Finding most early-game dominate player. 

In [5]:
zcols = ["z_gd10", "z_gd15","z_gd20",
        "z_csd10", "z_csd15", "z_csd20", 
         "z_xpd10", "z_xpd15", "z_xpd20"
        ]

df = pd.read_sql(
    f"SELECT playername, position, teamname, n_games, {', '.join(zcols)} FROM lane_dominance_z",
    engine,
)

In [6]:
df.head(5)

,playername,position,teamname,n_games,z_gd10,z_gd15,z_gd20,z_csd10,z_csd15,z_csd20,z_xpd10,z_xpd15,z_xpd20
0,Zven,bot,Cloud9,18,0.855,-0.001,0.029,2.113,2.021,1.755,1.511,1.570,1.284
1,deokdam,bot,"Dplus Kia, KT Rolster",28,1.082,0.637,0.806,-0.084,-0.462,-1.023,0.437,0.229,-0.229
2,huanfeng,bot,Suning,19,0.125,-0.398,0.424,-0.563,-0.395,0.468,0.359,0.701,1.576
3,Hans Sama,bot,G2 Esports,10,1.070,0.550,0.365,0.562,-0.128,0.082,0.546,-0.278,-0.503
4,Doggo,bot,CTBC Flying Oyster,12,0.989,1.626,1.596,0.039,0.997,1.022,0.331,0.631,0.779


In [7]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

In [8]:
df.shape

(127, 13)

In [9]:
df_cleaned = df.dropna()

In [11]:
X = df_cleaned[zcols].to_numpy()

In [12]:
pca = PCA(n_components = len(zcols)).fit(X)

In [13]:
first_component = pca.components_[0] * np.sign(pca.components_[0].sum()) 

In [14]:
first_component

array([0.3139175 , 0.332937  , 0.33336859, 0.33717293, 0.3526729 ,
       0.34195822, 0.31510264, 0.3491138 , 0.3214107 ])

In [15]:
print(pd.Series(first_component, index=zcols).round(3))

z_gd10     0.314
z_gd15     0.333
z_gd20     0.333
z_csd10    0.337
z_csd15    0.353
z_csd20    0.342
z_xpd10    0.315
z_xpd15    0.349
z_xpd20    0.321
dtype: float64


In [16]:
print(pd.Series(pca.explained_variance_ratio_, 
                index = [f'Component {i + 1}' for i in range(len(first_component))]).round(3))

Component 1    0.702
Component 2    0.106
Component 3    0.068
Component 4    0.060
Component 5    0.031
Component 6    0.018
Component 7    0.008
Component 8    0.004
Component 9    0.002
dtype: float64


In [17]:
df_cleaned = df_cleaned.copy()

In [18]:
df_cleaned["mean_lane_index"] = df_cleaned[zcols].mean(axis = 1)

In [19]:
df_cleaned["pca_lane_index"] = X @ first_component / np.sqrt(pca.explained_variance_[0])

In [24]:
lane_diff_df = (
    df_cleaned
    .rename(columns={"position": "roles"})
    .set_index("playername")[["roles", "teamname", "mean_lane_index", "pca_lane_index"]]
)

In [27]:
lane_diff_df.sort_values(by = "mean_lane_index", ascending = False).head(10)

,roles,teamname,mean_lane_index,pca_lane_index
playername,,,,
Blaber,jng,Cloud9,1.898444,2.317389
Chovy,mid,"Gen.G, Hanwha Life Esports, Kiwoom DRX",1.496556,1.818291
Gumayusi,bot,T1,1.322333,1.611500
Larssen,mid,Rogue,1.319000,1.594036
Hylissang,sup,Fnatic,1.273444,1.518648
Zven,bot,Cloud9,1.237444,1.512174
Xiaohu,top,Royal Never Give Up,1.138222,1.393636
TheShy,top,Weibo Gaming,1.099667,1.345724
Juhan,jng,Kiwoom DRX,0.954111,1.127356


In [28]:
lane_diff_df.sort_values(by = "pca_lane_index", ascending = False).head(10)

,roles,teamname,mean_lane_index,pca_lane_index
playername,,,,
Blaber,jng,Cloud9,1.898444,2.317389
Chovy,mid,"Gen.G, Hanwha Life Esports, Kiwoom DRX",1.496556,1.818291
Gumayusi,bot,T1,1.322333,1.611500
Larssen,mid,Rogue,1.319000,1.594036
Hylissang,sup,Fnatic,1.273444,1.518648
Zven,bot,Cloud9,1.237444,1.512174
Xiaohu,top,Royal Never Give Up,1.138222,1.393636
TheShy,top,Weibo Gaming,1.099667,1.345724
Juhan,jng,Kiwoom DRX,0.954111,1.127356


## Lane Dominance of Chovy, Faker, Guma, Keria, Zeus

In [40]:
ranked = lane_diff_df.copy()

# higher index = better, so rank descending; method="min" gives tied players the same rank
ranked["mean_rank"]         = ranked["mean_lane_index"].rank(ascending=False, method="min").astype(int)
ranked["pca_rank"]          = ranked["pca_lane_index"].rank(ascending=False, method="min").astype(int)
ranked["mean_rank_in_role"] = ranked.groupby("roles")["mean_lane_index"].rank(ascending=False, method="min").astype(int)
ranked["pca_rank_in_role"]  = ranked.groupby("roles")["pca_lane_index"].rank(ascending=False, method="min").astype(int)
ranked["n_in_role"]         = ranked.groupby("roles")["mean_lane_index"].transform("size")

# sort by mean_lane_index, as requested
ranked = ranked.sort_values("mean_lane_index", ascending=False)

In [41]:
zeus = ranked.loc[ranked.index == "Zeus"]
print(zeus.T)

playername                            Zeus
roles                                  top
teamname           Hanwha Life Esports, T1
mean_lane_index                   0.904333
pca_lane_index                    1.096966
mean_rank                               10
pca_rank                                10
mean_rank_in_role                        3
pca_rank_in_role                         3
n_in_role                               28


In [42]:
chovy = ranked.loc[ranked.index == "Chovy"]
print(chovy.T)

playername                                          Chovy
roles                                                 mid
teamname           Gen.G, Hanwha Life Esports, Kiwoom DRX
mean_lane_index                                  1.496556
pca_lane_index                                   1.818291
mean_rank                                               2
pca_rank                                                2
mean_rank_in_role                                       1
pca_rank_in_role                                        1
n_in_role                                              21


In [43]:
guma = ranked.loc[ranked.index == "Gumayusi"]
print(guma.T)

playername         Gumayusi
roles                   bot
teamname                 T1
mean_lane_index    1.322333
pca_lane_index       1.6115
mean_rank                 3
pca_rank                  3
mean_rank_in_role         1
pca_rank_in_role          1
n_in_role                24


In [44]:
keria = ranked.loc[ranked.index == "Keria"]
print(keria.T)

playername                  Keria
roles                         sup
teamname           Kiwoom DRX, T1
mean_lane_index          0.602778
pca_lane_index           0.714481
mean_rank                      27
pca_rank                       27
mean_rank_in_role               5
pca_rank_in_role                5
n_in_role                      28


In [45]:
faker = ranked.loc[ranked.index == "Faker"]
print(faker.T)

playername            Faker
roles                   mid
teamname                 T1
mean_lane_index   -0.279222
pca_lane_index    -0.336701
mean_rank                94
pca_rank                 94
mean_rank_in_role        14
pca_rank_in_role         14
n_in_role                21


# Chapter 2: Finding kappa for shrunk kp analysis @ 10, 15, 20 min, and whole game.

In [48]:
WINDOWS = {
    "10": ("p10", "t10"),
    "15": ("p15", "t15"),
    "20": ("p20", "t20"),
    "total": ("p_total", "t_total")
}

LOG_BOUNDS = (np.log(0.5), np.log(1e4))

tot = pd.read_sql("SELECT * FROM P_and_T", engine)

def get_data(w):
    """Returns {role: (P, T)} for window w."""
    p_col, t_col = WINDOWS[w]
    out = {}
    for role, g in tot.groupby("position"):

        d = g.dropna(subset=[p_col, t_col])
        d = d[d[t_col] > 0]

        if len(d) == 0:
            continue

        P = d[p_col].astype(int).to_numpy()
        T = d[t_col].astype(int).to_numpy()

        assert (P <= T).all(), (
            f"P > T for {role} at {w}"
        )

        out[role] = (P, T)

    return out


DATA = {w: get_data(w) for w in WINDOWS}
MU = {
    w: {
        r: P.sum() / T.sum()
        for r, (P, T) in DATA[w].items()
    }
    for w in WINDOWS
}
assert all(
    0 < m < 1
    for w in WINDOWS
    for m in MU[w].values()
), "mu outside (0, 1)"

def nll(log_k, P, T, m):
    k = np.exp(log_k)

    return -betabinom.logpmf(
        P, T, m * k, (1 - m) * k
    ).sum()


def fit(f):
    """Minimize f(log_kappa); returns (kappa, minimized nll)."""
    res = minimize_scalar(
        f,
        bounds=LOG_BOUNDS,
        method="bounded"
    )
    return float(np.exp(res.x)), float(res.fun)

def profile_ci(f, n=600):
    """Approximate 95% profile likelihood CI for kappa."""
    grid = np.linspace(*LOG_BOUNDS, n)
    vals = np.array([f(g) for g in grid])
    keep = grid[vals - vals.min() <= 1.92]
    return float(np.exp(keep.min())), float(np.exp(keep.max()))

rows = []
for w in WINDOWS:
    for role, (P, T) in DATA[w].items():
        m = MU[w][role]
        k, _ = fit(lambda lk: nll(lk, P, T, m))
        disp = (
            (((P - m * T) ** 2) / (m * (1 - m) * T)).sum()
            / (len(P) - 1)
            if len(P) > 1 else np.nan
        )
        rows.append(dict(
            position=role,
            window=w,
            n_players=len(P),
            mu=m,
            kappa=k,
            at_bound=(k < 0.6 or k > 9000),
            dispersion=disp,
            w_median=np.median(T) / (np.median(T) + k),
            implied_sd=np.sqrt(m * (1 - m) / (k + 1))
        ))
per_role = (
    pd.DataFrame(rows)
    .sort_values(["position", "window"])
    .round(4)
)
print(per_role.to_string(index=False))

def nll_shared(log_k, w):
    return sum(
        nll(log_k, P, T, MU[w][r])
        for r, (P, T) in DATA[w].items()
    )

shared = {}
for w in WINDOWS:
    k_sh, nll_sh = fit(lambda lk: nll_shared(lk, w))
    nll_sep = sum(
        fit(lambda lk: nll(lk, P, T, MU[w][r]))[1]
        for r, (P, T) in DATA[w].items()
    )
    lr = max(0, 2 * (nll_sh - nll_sep))
    df = len(DATA[w]) - 1
    lo, hi = profile_ci(
        lambda lk: nll_shared(lk, w)
    )
    shared[w] = dict(
        kappa=k_sh,
        ci_lo=lo,
        ci_hi=hi,
        LR=lr,
        df=df,
        p_value=chi2.sf(lr, df) if df > 0 else np.nan
    )
shared_df = pd.DataFrame(shared).T.round(3)
print(shared_df)

KAPPA_MODE = "shared"     

def kappa_for(role, w):
    if KAPPA_MODE == "shared":
        return round(shared[w]["kappa"])
    return round(
        float(
            per_role.query(
                "position == @role and window == @w"
            ).kappa.iloc[0]
        ),
        1
    )

common_roles = set.intersection(
    *(set(DATA[w]) for w in WINDOWS)
)
lines = []
for role in sorted(common_roles):
    vals = []
    for w in WINDOWS:
        vals += [
            round(MU[w][role], 4),
            kappa_for(role, w)
        ]
    lines.append(
        f"('{role}', " + ", ".join(str(v) for v in vals) + ")"
    )
print(
    "INSERT INTO kp_params VALUES\n"
    + ",\n".join(lines)
    + ";"
)


position window  n_players     mu     kappa  at_bound  dispersion  w_median  implied_sd
     bot     10         24 0.4883  115.8889     False      1.6093    0.2863      0.0462
     bot     15         24 0.5008   69.0387     False      2.4214    0.5305      0.0597
     bot     20         24 0.5615   65.8362     False      3.0509    0.6523      0.0607
     bot  total         24 0.6813  119.3952     False      3.6327    0.7105      0.0425
     jng     10         26 0.7199 9999.9584      True      0.8373    0.0043      0.0045
     jng     15         26 0.7224 1945.1732     False      0.9961    0.0393      0.0102
     jng     20         26 0.7238 9999.9490      True      0.8894    0.0121      0.0045
     jng  total         26 0.7178  408.9001     False      1.8877    0.4051      0.0222
     mid     10         21 0.4695   30.8097     False      3.0526    0.6040      0.0885
     mid     15         21 0.5247   59.0062     False      2.9692    0.6040      0.0645
     mid     20         21 0.582

# Chapter 3. Finding tau^2 values for shrinking win rate above expected.

In [47]:
SUBSETS = ("carry", "carried", "aligned")

def fit_tau2(subset):
    pp = pd.read_sql(f"""
        SELECT playerid, position,
               SUM(n_games)::float                                                AS n,
               SUM(wins_above_expected)::float                                    AS r,
               (SUM(n_games * others_wr * (1 - others_wr)) / SUM(n_games))::float AS v
        FROM off_diagonals
        WHERE subset = '{subset}'
        GROUP BY playerid, position
    """, engine)

    pp["d"]   = pp.r / pp.n          # raw per-game effect
    pp["se2"] = pp.v / pp.n          # its sampling variance

    def nll(log_tau2):
        var = np.exp(log_tau2) + pp.se2
        return 0.5 * (np.log(var) + pp.d ** 2 / var).sum()

    lo, hi = np.log(1e-7), np.log(1.0)
    res  = minimize_scalar(nll, bounds=(lo, hi), method="bounded")
    tau2 = float(np.exp(res.x))
    lr   = 2 * (nll(lo) - res.fun)                       # likelihood-ratio statistic vs tau^2 = 0

    return dict(subset=subset,
                players=len(pp),
                tau2=tau2,
                tau=tau2 ** 0.5,
                mean_k=(pp.v / tau2).mean(),
                at_bound=bool(res.x < lo + 0.1),                          # True = no usable spread
                lr=lr,
                p_value=1.0 if lr <= 0 else 0.5 * chi2.sf(lr, 1),         # boundary-corrected: tau^2 = 0 is on the edge
                dispersion=(pp.d ** 2 / pp.se2).sum() / len(pp))          # diluted by low-game players; read with the LR test

fits = pd.DataFrame([fit_tau2(s) for s in SUBSETS])
print(fits.round(5).to_string(index=False))

# lines to paste into the params CTE
print("\n" + "\nUNION ALL\n".join(
    f"SELECT '{r.subset}' AS subset, {r.tau2:.5f}::numeric AS tau2" for r in fits.itertuples()))

 subset  players    tau2     tau    mean_k  at_bound       lr  p_value  dispersion
  carry      324 0.00573 0.07572  35.85745     False  7.17771  0.00369     1.05774
carried      311 0.00090 0.03005 231.13237     False  0.25843  0.30560     1.02451
aligned      337 0.00255 0.05051  49.40503     False 10.54239  0.00058     1.17161

SELECT 'carry' AS subset, 0.00573::numeric AS tau2
UNION ALL
SELECT 'carried' AS subset, 0.00090::numeric AS tau2
UNION ALL
SELECT 'aligned' AS subset, 0.00255::numeric AS tau2
